# 01 · DataFrame basics

In this lesson: creating DataFrames, inspecting them, selecting/adding/removing columns, filtering and
sorting rows. These are the bread and butter of every Spark job.

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

## Creating a DataFrame from Python data

`spark.createDataFrame(data, schema)` turns a list of tuples into a DataFrame. The `schema` string names
each column and gives its type. Python `None` becomes Spark `null` (a missing value).

In [2]:
people = spark.createDataFrame(
    [("Alice", 34, "London"), ("Bob", 45, "Paris"), ("Cara", 29, "London"),
     ("Dan", 52, "Berlin"), ("Eve", 38, "Paris"), ("Finn", 23, None)],
    schema="name string, age int, city string",
)
people.show()

+-----+---+------+
| name|age|  city|
+-----+---+------+
|Alice| 34|London|
|  Bob| 45| Paris|
| Cara| 29|London|
|  Dan| 52|Berlin|
|  Eve| 38| Paris|
| Finn| 23|  NULL|
+-----+---+------+



Other ways: a list of `Row` objects or dicts (Spark infers the types), or from pandas with `spark.createDataFrame(pandas_df)`.

In [3]:
from pyspark.sql import Row
import pandas as pd

spark.createDataFrame([Row(x=1, y="a"), Row(x=2, y="b")]).show()
spark.createDataFrame(pd.DataFrame({"x": [1, 2], "y": ["a", "b"]})).show()

+---+---+
|  x|  y|
+---+---+
|  1|  a|
|  2|  b|
+---+---+



+---+---+
|  x|  y|
+---+---+
|  1|  a|
|  2|  b|
+---+---+



## Inspecting a DataFrame

In [4]:
people.printSchema()
print("columns:", people.columns)
print("dtypes: ", people.dtypes)
print("rows:   ", people.count())

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)

columns: ['name', 'age', 'city']
dtypes:  [('name', 'string'), ('age', 'int'), ('city', 'string')]


rows:    6


In [5]:
people.describe().show()     # quick summary statistics

+-------+-----+------------------+------+
|summary| name|               age|  city|
+-------+-----+------------------+------+
|  count|    6|                 6|     5|
|   mean| NULL|36.833333333333336|  NULL|
| stddev| NULL|10.571975532825768|  NULL|
|    min|Alice|                23|Berlin|
|    max| Finn|                52| Paris|
+-------+-----+------------------+------+



In [6]:
people.show(2)                          # only 2 rows
people.show(truncate=False)             # don't cut long strings at 20 chars
people.show(2, vertical=True)           # one field per line — handy for wide tables

+-----+---+------+
| name|age|  city|
+-----+---+------+
|Alice| 34|London|
|  Bob| 45| Paris|
+-----+---+------+
only showing top 2 rows
+-----+---+------+
|name |age|city  |
+-----+---+------+
|Alice|34 |London|
|Bob  |45 |Paris |
|Cara |29 |London|
|Dan  |52 |Berlin|
|Eve  |38 |Paris |
|Finn |23 |NULL  |
+-----+---+------+

-RECORD 0------
 name | Alice  
 age  | 34     
 city | London 
-RECORD 1------
 name | Bob    
 age  | 45     
 city | Paris  
only showing top 2 rows


## Bringing data back into Python

| Method | Returns | Careful? |
|---|---|---|
| `df.first()` | one `Row` | fine |
| `df.take(n)` | list of `n` Rows | fine for small n |
| `df.collect()` | list of **all** Rows | pulls the *entire* dataset to the driver — can crash on big data |
| `df.toPandas()` | pandas DataFrame of **all** rows | same warning |

A `Row` behaves like a tuple *and* like an object with attributes.

In [7]:
rows = people.take(2)
print(rows)
print(rows[0]["name"], rows[0].age, rows[0][2])
people.limit(3).toPandas()       # limit first, then convert — safe habit

[Row(name='Alice', age=34, city='London'), Row(name='Bob', age=45, city='Paris')]
Alice 34 London


,name,age,city
0,Alice,34,London
1,Bob,45,Paris
2,Cara,29,London


## Selecting columns

There are four ways to refer to a column. They're mostly interchangeable:

In [8]:
people.select("name", "age").show(2)                               # 1. a string
people.select(people.name, people["age"], F.col("city")).show(2)    # 2. attribute  3. bracket  4. F.col

+-----+---+
| name|age|
+-----+---+
|Alice| 34|
|  Bob| 45|
+-----+---+
only showing top 2 rows
+-----+---+------+
| name|age|  city|
+-----+---+------+
|Alice| 34|London|
|  Bob| 45| Paris|
+-----+---+------+
only showing top 2 rows


**Prefer `F.col("name")`.** It doesn't need a DataFrame variable, so it works in long chains where the
intermediate DataFrame has no name. Columns can be combined into **expressions**, and `.alias()` names the result:

In [9]:
people.select("name", (F.col("age") + 1).alias("age_next_year")).show()

+-----+-------------+
| name|age_next_year|
+-----+-------------+
|Alice|           35|
|  Bob|           46|
| Cara|           30|
|  Dan|           53|
|  Eve|           39|
| Finn|           24|
+-----+-------------+



## Adding, renaming and dropping columns

In [10]:
people2 = (people
           .withColumn("age_next_year", F.col("age") + 1)          # add (or replace) a column
           .withColumn("is_londoner", F.col("city") == "London")
           .withColumnRenamed("name", "first_name")
           .drop("age"))
people2.show()

# add several at once
people.withColumns({"age_x2": F.col("age") * 2, "country": F.lit("?")}).show(2)

+----------+------+-------------+-----------+
|first_name|  city|age_next_year|is_londoner|
+----------+------+-------------+-----------+
|     Alice|London|           35|       true|
|       Bob| Paris|           46|      false|
|      Cara|London|           30|       true|
|       Dan|Berlin|           53|      false|
|       Eve| Paris|           39|      false|
|      Finn|  NULL|           24|       NULL|
+----------+------+-------------+-----------+



+-----+---+------+------+-------+
| name|age|  city|age_x2|country|
+-----+---+------+------+-------+
|Alice| 34|London|    68|      ?|
|  Bob| 45| Paris|    90|      ?|
+-----+---+------+------+-------+
only showing top 2 rows


### DataFrames are immutable
None of the above changed `people`. Every transformation returns a **new** DataFrame; you capture it
by assigning to a variable (`people2 = ...`).

In [11]:
people.show(2)     # still the original columns

+-----+---+------+
| name|age|  city|
+-----+---+------+
|Alice| 34|London|
|  Bob| 45| Paris|
+-----+---+------+
only showing top 2 rows


## Filtering rows

`filter` and `where` are identical. Combine conditions with **`&` (and), `|` (or), `~` (not)** —
*not* Python's `and`/`or`/`not` — and wrap each condition in parentheses.

In [12]:
people.filter(F.col("age") > 35).show()
people.where("age > 35").show()                                         # a SQL string works too
people.filter((F.col("age") > 30) & (F.col("city") == "Paris")).show()
people.filter(F.col("city").isin("London", "Berlin")).show()
people.filter(F.col("city").isNull()).show()                             # NOT  == None
people.filter(F.col("name").startswith("E") | F.col("name").contains("nn")).show()

+----+---+------+
|name|age|  city|
+----+---+------+
| Bob| 45| Paris|
| Dan| 52|Berlin|
| Eve| 38| Paris|
+----+---+------+

+----+---+------+
|name|age|  city|
+----+---+------+
| Bob| 45| Paris|
| Dan| 52|Berlin|
| Eve| 38| Paris|
+----+---+------+



+----+---+-----+
|name|age| city|
+----+---+-----+
| Bob| 45|Paris|
| Eve| 38|Paris|
+----+---+-----+



+-----+---+------+
| name|age|  city|
+-----+---+------+
|Alice| 34|London|
| Cara| 29|London|
|  Dan| 52|Berlin|
+-----+---+------+



+----+---+----+
|name|age|city|
+----+---+----+
|Finn| 23|NULL|
+----+---+----+

+----+---+-----+
|name|age| city|
+----+---+-----+
| Eve| 38|Paris|
|Finn| 23| NULL|
+----+---+-----+



### Gotcha: comparisons with null

In Spark (as in SQL) `null != "London"` is not `True` — it's `null`, and `filter` drops it.
So Finn (city = null) **disappears** here:

In [13]:
people.filter(F.col("city") != "London").show()

+----+---+------+
|name|age|  city|
+----+---+------+
| Bob| 45| Paris|
| Dan| 52|Berlin|
| Eve| 38| Paris|
+----+---+------+



## Sorting, limiting, de-duplicating

In [14]:
people.orderBy("age").show(3)
people.orderBy(F.col("age").desc()).show(3)
people.orderBy(F.col("city").asc_nulls_last(), F.desc("age")).show()
people.select("city").distinct().show()
people.dropDuplicates(["city"]).show()     # keep one (arbitrary) row per city

+-----+---+------+
| name|age|  city|
+-----+---+------+
| Finn| 23|  NULL|
| Cara| 29|London|
|Alice| 34|London|
+-----+---+------+
only showing top 3 rows


+----+---+------+
|name|age|  city|
+----+---+------+
| Dan| 52|Berlin|
| Bob| 45| Paris|
| Eve| 38| Paris|
+----+---+------+
only showing top 3 rows
+-----+---+------+
| name|age|  city|
+-----+---+------+
|  Dan| 52|Berlin|
|Alice| 34|London|
| Cara| 29|London|
|  Bob| 45| Paris|
|  Eve| 38| Paris|
| Finn| 23|  NULL|
+-----+---+------+



+------+
|  city|
+------+
|London|
| Paris|
|Berlin|
|  NULL|
+------+



+-----+---+------+
| name|age|  city|
+-----+---+------+
| Finn| 23|  NULL|
|  Dan| 52|Berlin|
|Alice| 34|London|
|  Bob| 45| Paris|
+-----+---+------+



## Style tip: chain inside parentheses
```python
result = (df
          .filter(...)
          .withColumn(...)
          .select(...)
          .orderBy(...))
```
Each step on its own line is easy to read and easy to comment out while debugging.

### Exercise 1
Create `over_30`: the **name and age** of people **older than 30**, sorted **oldest first**.

In [15]:
over_30 = (people
           .filter(F.col("age") > 30)
           .select("name", "age")
           .orderBy(F.col("age").desc()))

In [16]:
# Check — run me after your answer
assert over_30.columns == ["name", "age"], f"columns should be ['name', 'age'], got {over_30.columns}"
assert [r.name for r in over_30.collect()] == ["Dan", "Bob", "Eve", "Alice"], "wrong rows or wrong order"
print("Correct!")

Correct!


### Exercise 2
Create `with_months`: all columns of `people` plus a new column **`age_months`** = age × 12.

In [17]:
with_months = people.withColumn("age_months", F.col("age") * 12)

In [18]:
# Check — run me after your answer
assert with_months.columns == ["name", "age", "city", "age_months"]
assert with_months.filter(F.col("name") == "Alice").first().age_months == 408
print("Correct!")

Correct!


### Exercise 3
Store in `n_cities` the number of **distinct, non-null** cities.

In [19]:
n_cities = people.filter(F.col("city").isNotNull()).select("city").distinct().count()

In [20]:
# Check — run me after your answer
assert n_cities == 3, f"expected 3, got {n_cities}"
print("Correct!")

Correct!


### Exercise 4
Create `not_paris`: everyone who does **not** live in Paris — **including** Finn, whose city is null.

*Hint:* either add an `isNull()` condition with `|`, or look up `eqNullSafe`.

In [21]:
not_paris = people.filter((F.col("city") != "Paris") | F.col("city").isNull())
# or: people.filter(~F.col("city").eqNullSafe("Paris"))

In [22]:
# Check — run me after your answer
assert sorted(r.name for r in not_paris.collect()) == ["Alice", "Cara", "Dan", "Finn"]
print("Correct!")

Correct!


## Recap
`createDataFrame` · `show` / `printSchema` / `count` · `select` + `F.col` + `alias` · `withColumn` / `withColumnRenamed` / `drop`
· `filter` with `& | ~` · nulls need `isNull` · `orderBy`, `distinct`, `dropDuplicates` · DataFrames are immutable.